<a href="https://colab.research.google.com/github/vad-source/NLPAPP/blob/main/SENTIMENT/NLPAPP_Deep_LearningOpen_SA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP APPLICATIONS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

In [ ]:
!pip -q install transformers torch sentencepiece spacy pyabsa presidio-analyzer presidio-anonymizer

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.2/574.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart r

In [ ]:
import re
import uuid
import unicodedata

import spacy

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

from transformers import pipeline

In [ ]:
nlp = spacy.load("en_core_web_sm")

pii_analyzer = AnalyzerEngine()
pii_anonymizer = AnonymizerEngine()

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 0 : Ingestion (source Data)

## 1 : Preprocessor

### 1.1 : Inbound Guardrails

In [ ]:
class InboundGuardrails:
    MAX_LEN = 1000
    BLOCK_PATTERNS = [
        r"<script",
        r"DROP TABLE",
        r"DELETE FROM",
        r"IGNORE PREVIOUS INSTRUCTIONS"
    ]
    def run(self, text):
        if not text:
            raise ValueError("Empty Payload")
        if len(text) > self.MAX_LEN:
            raise ValueError("Payload Too Large")
        for pattern in self.BLOCK_PATTERNS:
            if re.search(pattern, text, re.IGNORECASE):
                raise ValueError(f"Blocked Pattern: {pattern}")
        return text

### 1.2 : Compliance Check

In [ ]:
class CompliancePII:
    def run(self, text):
        findings = pii_analyzer.analyze(text=text,language="en")
        return pii_anonymizer.anonymize(text=text,analyzer_results=findings).text

### 1.3 : Normalization

In [ ]:
class TextNormalizer:
    def run(self, text):
        text = unicodedata.normalize("NFKC",text)
        text = text.lower()
        text = re.sub(r"\s+"," ",text)
        return text.strip()

In [ ]:
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
normalizeT = TextNormalizer()
print(normalizeT.run(complianceT.run(guardT.run(text))))

can’t even comment. customer service was not very terrible !!!! . contact me: [email] [phone]. use [apikey]


## 2 : Representation & Processing

In [ ]:
class DocumentVectorizer:
    def run(self, text):
        doc = nlp(text)
        return {"raw_text": text,"tokens":[t.text for t in doc],"pos_tags":[(t.text, t.pos_) for t in doc], "spacy_doc": doc }

In [ ]:
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
vectorizeT = DocumentVectorizer()
print(vectorizeT.run(normalizeT.run(complianceT.run(guardT.run(text)))))

{'raw_text': 'can’t even comment. customer service was not very terrible !!!! . contact me: [email] [phone]. use [apikey]', 'tokens': ['can', '’', 't', 'even', 'comment', '.', 'customer', 'service', 'was', 'not', 'very', 'terrible', '!', '!', '!', '!', '.', 'contact', 'me', ':', '[', 'email', ']', '[', 'phone', ']', '.', 'use', '[', 'apikey', ']'], 'pos_tags': [('can', 'MD'), ('’', 'VB'), ('t', 'VB'), ('even', 'RB'), ('comment', 'NN'), ('.', '.'), ('customer', 'NN'), ('service', 'NN'), ('was', 'VBD'), ('not', 'RB'), ('very', 'RB'), ('terrible', 'JJ'), ('!', '.'), ('!', '.'), ('!', '.'), ('!', '.'), ('.', '.'), ('contact', 'VB'), ('me', 'PRP'), (':', ':'), ('[', 'JJ'), ('email', 'NN'), (']', 'NNP'), ('[', 'NNP'), ('phone', 'NN'), (']', 'NN'), ('.', '.'), ('use', 'NN'), ('[', 'JJ'), ('apikey', 'NN'), (']', 'NN')], 'spacy_doc': can’t even comment. customer service was not very terrible !!!! . contact me: [email] [phone]. use [apikey]}


## 3 : Analyser (Inference Engine)

### A : Document Level

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

In [ ]:
bart_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [ ]:
text = """
The launch started strongly.
Customer adoption dropped.
The latest release improved retention.
"""

In [ ]:
sentence_classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1855: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ca97d98dc50>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ca97b7fecf0>


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/roberta/tokenization_roberta.py:144: DeprecationWarning: Deprecated in 0.9.0: BPE.__init__ will not create from files anymore, try `BPE.from_file` instead
  BPE(


In [ ]:
#print(summary)

The launch started strongly. Customer adoption dropped. The latest release improved retention. The launch started strong.Customer adoption dropped, but retention improved. The release started strongly, but dropped. It improved retention, but the latest release increased retention. It also improved adoption, but it dropped retention.


In [ ]:
class DocumentLevelSeq2SeqEngine:
    def run(self, vec):
        text = vec["raw_text"]
        inputs = bart_tokenizer( text, return_tensors="pt",truncation=True,max_length=24)
        summary_ids = bart_model.generate(**inputs, max_new_tokens=120)
        summary = bart_tokenizer.decode(summary_ids[0],skip_special_tokens=True)
        sentiment = sentence_classifier(text)
        return {"summary": summary, "sentiment_projection":  sentiment}

#### B : Sentence Level

In [ ]:
class SentenceLevelGRUEngine:
    def run(self, vec):
        text = vec["raw_text"]
        result = sentence_classifier(text)
        return result

#### C : Aspect Level

In [ ]:
from pyabsa import AspectTermExtraction as ATEPC

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


No CUDA GPU found in your device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-05-31 08:17:38] (2.4.2) PyABSA(2.4.2): If your code crashes on Colab, please use the GPU runtime. Then run "pip install pyabsa[dev] -U" and restart the kernel.
Or if it does not work, you can use v1.x versions, e.g., pip install pyabsa<2.0 -U




Try to downgrade transformers<=4.29.0.






In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
import torch

In [ ]:
tokenizer = AutoTokenizer.from_pretrained( "google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained( "google/flan-t5-base")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
class AspectLevelPointerGeneratorEngine:
    def __init__(self):
        self.tokenizer = tokenizer
        self.model = model
    def run(self, vec):
        text = vec["raw_text"]
        prompt = f"""
Extract aspects and sentiment.

Text:
{text}

Format:
aspect : sentiment"""

        inputs = self.tokenizer(  prompt, return_tensors="pt", truncation=True)
        outputs = self.model.generate(**inputs, max_new_tokens=20)
        result = self.tokenizer.decode(outputs[0],skip_special_tokens=True)
        return {"aspects": result}

In [ ]:
doc = nlp(text)

aspects = [
    chunk.text
    for chunk in doc.noun_chunks
]

## 4 : Post Processor

### 4.1 Explainer

In [ ]:
class ExplainabilityTelemetry:
    def run(self, result):
        result["telemetry"] = {"trace_id":str(uuid.uuid4()),"model_version":"rule_based_v1"}
        return result

### 4.2 : Outbound Guardrails

In [ ]:
class OutboundGuardrails:
    def run(self, result):
        result["status"] = "SUCCESS"
        return result

## Evaluator

## Pipeline

In [ ]:
def run_document_pipeline(text):
    text = InboundGuardrails().run(text)
    text = CompliancePII().run(text)
    text = TextNormalizer().run(text)
    vec = DocumentVectorizer().run(text)
    result = DocumentLevelSeq2SeqEngine().run(vec)
    result = ExplainabilityTelemetry().run(result)
    result =  OutboundGuardrails().run(result)
    return result

In [ ]:
def run_sentence_pipeline(text):
    text = InboundGuardrails().run(text)
    text = CompliancePII().run(text)
    text = TextNormalizer().run(text)
    vec = DocumentVectorizer().run(text)
    result = SentenceLevelGRUEngine().run(vec)
    result =ExplainabilityTelemetry().run({"prediction": result})
    result = OutboundGuardrails().run(result)
    return result

In [ ]:
def run_aspect_pipeline(text):
    text = InboundGuardrails().run(text)
    text = CompliancePII().run(text)
    text = TextNormalizer().run(text)
    vec = DocumentVectorizer().run(text)
    result =AspectLevelPointerGeneratorEngine().run(vec)
    result = ExplainabilityTelemetry().run({"prediction": result})
    result = OutboundGuardrails().run(result)
    return result

In [ ]:
doc = """
The launch started strongly.
Customer adoption dropped.
The latest release improved retention.
"""

run_document_pipeline(doc)

{'summary': 'the launch started strongly. customer adoption dropped. The latest release improved retention. The release of the latest version of the software is expected to be released in the next few weeks. The launch of the new software will be followed by the release of a new version of it in the coming months.',
 'sentiment_projection': [{'label': 'positive', 'score': 0.6727660894393921}],
 'telemetry': {'trace_id': '6aa83fea-5f26-440a-980b-bc6f01bb9a45',
  'model_version': 'rule_based_v1'},
 'status': 'SUCCESS'}

In [ ]:
run_sentence_pipeline(
    "Customer service was not very terrible."
)

{'prediction': [{'label': 'positive', 'score': 0.4864392578601837}],
 'telemetry': {'trace_id': 'cfc7ad44-4f59-4212-9b48-2f4332be5555',
  'model_version': 'rule_based_v1'},
 'status': 'SUCCESS'}

In [ ]:
run_aspect_pipeline(
    "Battery life is excellent but fan noise is annoying."
)

{'prediction': {'aspects': 'negative'},
 'telemetry': {'trace_id': 'e696f857-60bd-4829-bbf1-305f554cb3bc',
  'model_version': 'rule_based_v1'},
 'status': 'SUCCESS'}

In [ ]:
train_data = [
    (
        "battery life is excellent but fan noise is annoying",
        "battery life ; fan noise"
    ),
    (
        "screen quality is amazing and speakers are weak",
        "screen quality ; speakers"
    ),
    (
        "keyboard feels great but trackpad is terrible",
        "keyboard ; trackpad"
    ),
    (
        "camera performance is excellent",
        "camera performance"
    ),
    (
        "display is sharp and battery backup is good",
        "display ; battery backup"
    )
]

In [ ]:
from collections import Counter
SPECIAL = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
counter = Counter()

for x, y in train_data:
    counter.update(x.split())
    counter.update(y.split())

vocab = SPECIAL + sorted(counter.keys())

word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

In [ ]:
def encode(text):
    ids = [word2idx.get(t, word2idx["<UNK>"])for t in text.split()]
    return ids

def decode(ids):
    words = []
    for i in ids:
        token = idx2word[i]
        if token == "<EOS>":
            break
        words.append(token)
    return " ".join(words)

In [ ]:
import torch
pairs = []
for src, tgt in train_data:
    src_ids = encode(src)
    tgt_ids = ([word2idx["<SOS>"]]+ encode(tgt)+ [word2idx["<EOS>"]])
    pairs.append((torch.tensor(src_ids),torch.tensor(tgt_ids)))

In [ ]:
import torch.nn as nn
class Encoder(nn.Module):
    def __init__(self,vocab_size,emb_dim=64,hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.bilstm = nn.LSTM( emb_dim,hidden,bidirectional=True, batch_first=True)
    def forward(self, x):
        emb = self.embedding(x)
        outputs, (h,c) = self.bilstm(emb)
        return outputs

In [ ]:
class PointerAttention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.linear = nn.Linear(hidden*2,hidden*2)

    def forward(self,decoder_state,encoder_outputs):
        scores = torch.matmul(encoder_outputs, decoder_state.unsqueeze(-1)).squeeze(-1)
        weights = torch.softmax( scores,dim=-1)
        context = torch.sum(encoder_outputs * weights.unsqueeze(-1),dim=1)
        return context, weights

In [ ]:
class Decoder(nn.Module):
    def __init__(self,vocab_size,hidden=128,emb_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,emb_dim)
        self.gru = nn.GRU(emb_dim + hidden*2,hidden*2,batch_first=True)
        self.pointer = PointerAttention(hidden)
        self.fc = nn.Linear(hidden*2,vocab_size)

    def forward(self,token, hidden,encoder_outputs):
        emb = self.embedding(token)
        context, _ = self.pointer(hidden.squeeze(0),encoder_outputs)
        gru_in = torch.cat([emb, context.unsqueeze(1)],dim=-1)
        out, hidden = self.gru(gru_in, hidden )
        logits = self.fc(out.squeeze(1))
        return logits, hidden

In [ ]:
class AspectPointerNet(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = Encoder( vocab_size)
        self.decoder = Decoder( vocab_size)
    def forward(self,src,tgt):
        enc_out = self.encoder(src)
        hidden = enc_out[:,-1,:].unsqueeze(0)
        loss = 0
        criterion = nn.CrossEntropyLoss()
        token = tgt[:,0].unsqueeze(1)
        for t in range(1,tgt.size(1)):
            logits, hidden = self.decoder(token,hidden,enc_out)
            loss += criterion(logits,tgt[:,t])
            token = tgt[:,t].unsqueeze(1)
        return loss

In [ ]:
model = AspectPointerNet(len(vocab))
optimizer = torch.optim.Adam( model.parameters(),lr=0.001)

In [ ]:
for epoch in range(300):
    total = 0
    for src,tgt in pairs:
        src = src.unsqueeze(0)
        tgt = tgt.unsqueeze(0)
        optimizer.zero_grad()
        loss = model(src,tgt)
        loss.backward()
        optimizer.step()
        total += loss.item()
    if epoch % 50 == 0:
        print(epoch,total)

0 77.65433883666992
50 0.07554868888109922
100 0.02587879146449268
150 0.013559494633227587
200 0.008450409746728837
250 0.0057870837626978755


In [ ]:
def predict(text):
    src = torch.tensor(encode(text)).unsqueeze(0)
    enc_out = model.encoder(src)
    hidden = enc_out[:,-1,:].unsqueeze(0)
    token = torch.tensor([[word2idx["<SOS>"]]])
    output = []
    for _ in range(20):
        logits, hidden = model.decoder(token,hidden, enc_out)
        pred = logits.argmax(-1)
        idx = pred.item()
        if idx == word2idx["<EOS>"]:
            break
        output.append(idx)
        token = pred.unsqueeze(1)
    return decode(output)

In [ ]:
predict(
    "battery life is excellent but fan noise is annoying"
)

'battery life ; fan noise'

In [ ]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def aspect_sentiment_analyzer(review, aspects):

    results = []

    for aspect in aspects:

        prompt = f"""
        Review: {review}

        Aspect: {aspect}

        Question: What is the sentiment toward this aspect?
        """

        pred = sentiment_model(prompt)[0]

        results.append({
            "aspect": aspect,
            "sentiment": pred["label"],
            "confidence": pred["score"]
        })

    return results

In [ ]:
def run_absa_pipeline(review):

    # Step 1: Extract aspects (from pointer model)
    aspect_text = predict(review)

    aspects = [
        a.strip()
        for a in aspect_text.split(";")
    ]

    # Step 2: sentiment per aspect
    results = aspect_sentiment_analyzer(
        review,
        aspects
    )

    return results

In [ ]:
run_absa_pipeline(
    "Battery life is excellent but fan noise is terrible."
)

[{'aspect': 'battery life',
  'sentiment': 'negative',
  'confidence': 0.5595769286155701},
 {'aspect': 'fan noise',
  'sentiment': 'negative',
  'confidence': 0.7042419910430908}]